
## Xây Dựng Cấu Hình Kiến Trúc B1 = YOLOv8n + P2

### 1. Mục tiêu
- **Kiến trúc B1 ($YOLOv8n + P_2$)**: Bổ sung tầng phát hiện phân giải cao $P_2$ (stride 4, kích thước $160 \times 160$ tại đầu vào $640 \times 640$) nhằm tăng cường khả năng trích xuất và bảo toàn đặc trưng hình học cho các khuyết tật PCB siêu nhỏ (tiny/small defects).
- **Nguyên tắc cô lập biến (Isolation of Variables)**: Thử nghiệm $B_1$ **chỉ thay đổi duy nhất kiến trúc mạng**.
- **Bảo toàn các điều kiện thực nghiệm**:
  - Giữ nguyên tập phân chia dữ liệu (`train.txt`, `val.txt`, `split_manifest.json`).
  - Giữ nguyên số ngẫu nhiên `seed: 41`.
  - Giữ nguyên công thức tăng cường dữ liệu (`mosaic: 1.0`, `fliplr: 0.5`, v.v.).
  - Giữ nguyên hàm mất mát (loss function: CIoU + DFL + BCE) và số epoch (20 epochs giống $B_0$).
  - Tuyệt đối không can thiệp ngưỡng dự đoán (threshold) và **không dùng tập test** để tuning.

### 2. Import thư viện
Import các thư viện chuẩn cho việc xử lý cấu hình mô hình: `pathlib`, `yaml`, `shutil`, `ultralytics`.

In [1]:
import os
from pathlib import Path
import shutil
import yaml
import ultralytics
from ultralytics import YOLO

def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
print(f"Project root: {ROOT}")
print(f"Ultralytics version: {ultralytics.__version__}")

Project root: /Users/Project/kltn-pcb
Ultralytics version: 8.4.142


### 3. Định vị thư mục chứa cấu hình YAML của Ultralytics
- Tìm đường dẫn file gốc `ultralytics.__file__`.
- Xác định vị trí thư mục định nghĩa kiến trúc `cfg/models/v8`.
- Liệt kê danh sách các file YAML khả dụng (`yolov8.yaml`, `yolov8-p2.yaml`, v.v.).

In [2]:
ultralytics_file = Path(ultralytics.__file__).resolve()
models_v8_dir = ultralytics_file.parent / "cfg" / "models" / "v8"

print(f"ultralytics.__file__: {ultralytics_file}")
print(f"Thư mục cfg/models/v8 : {models_v8_dir}")
print(f"Tồn tại? : {models_v8_dir.exists()}")

if models_v8_dir.exists():
    available_yamls = sorted([f.name for f in models_v8_dir.glob("*.yaml")])
    print(f"\nTìm thấy {len(available_yamls)} file YAML kiến trúc:")
    for yml in available_yamls:
        marker = " <== [Mục tiêu đối chiếu]" if yml in ["yolov8.yaml", "yolov8-p2.yaml"] else ""
        print(f"  - {yml}{marker}")
else:
    print("CẢNH BÁO: Không tìm thấy thư mục cfg/models/v8 trong ultralytics package!")

ultralytics.__file__: /opt/anaconda3/lib/python3.13/site-packages/ultralytics/__init__.py
Thư mục cfg/models/v8 : /opt/anaconda3/lib/python3.13/site-packages/ultralytics/cfg/models/v8
Tồn tại? : True

Tìm thấy 19 file YAML kiến trúc:
  - yoloe-v8-seg.yaml
  - yoloe-v8.yaml
  - yolov8-cls-resnet101.yaml
  - yolov8-cls-resnet50.yaml
  - yolov8-cls.yaml
  - yolov8-ghost-p2.yaml
  - yolov8-ghost-p6.yaml
  - yolov8-ghost.yaml
  - yolov8-obb.yaml
  - yolov8-p2.yaml <== [Mục tiêu đối chiếu]
  - yolov8-p6.yaml
  - yolov8-pose-p6.yaml
  - yolov8-pose.yaml
  - yolov8-rtdetr.yaml
  - yolov8-seg-p6.yaml
  - yolov8-seg.yaml
  - yolov8-world.yaml
  - yolov8-worldv2.yaml
  - yolov8.yaml <== [Mục tiêu đối chiếu]


### 4. Đọc YAML mẫu và đối chiếu kiến trúc B0 vs B1

#### Đối chiếu mặt lý thuyết giữa B0 và B1:
- **Baseline B0 (`yolov8n.pt`)**:
  - Cấu trúc FPN/PAN tiêu chuẩn trích xuất đặc trưng tại 3 mức phân giải: $P_3$ (stride 8, $80 \times 80$), $P_4$ (stride 16, $40 \times 40$), $P_5$ (stride 32, $20 \times 20$).
  - Đầu ra phát hiện `Detect` nhận 3 feature maps: `[P3, P4, P5]`.
  - Hạn chế: Khuyết tật PCB có kích thước rất nhỏ (nhiều box $< 20 \times 20$ pixel) khi qua các phép downsampling sâu xuống $P_4, P_5$ bị tiêu biến thông tin vị trí không gian.

- **Thực nghiệm B1 ($YOLOv8n + P_2$)**:
  - Nhánh FPN được mở rộng thêm một bước upsample ngược về tầng đặc trưng phân giải cao $P_2$ (stride 4, $160 \times 160$ pixel).
  - Đầu ra phát hiện `Detect` nhận 4 feature maps: `[P2, P3, P4, P5]`.
  - Cung cấp lưới biểu diễn chi tiết hơn gấp 4 lần so với $P_3$, giúp mô hình phát hiện chính xác các vết xước nhỏ (`mouse_bite`, `spur`) và mối hở/chập siêu mảnh (`open_circuit`, `short`).

In [3]:
yolov8_path = models_v8_dir / "yolov8.yaml"
yolov8_p2_path = models_v8_dir / "yolov8-p2.yaml"

if yolov8_path.exists():
    with open(yolov8_path, "r", encoding="utf-8") as f:
        b0_yaml_content = f.read()
    print(f"=== {yolov8_path.name} (Đầu vào của Baseline B0) ===")
    print(b0_yaml_content[:600] + "\n... [xem tiếp trong file] ...\n")

if yolov8_p2_path.exists():
    with open(yolov8_p2_path, "r", encoding="utf-8") as f:
        p2_yaml_content = f.read()
    print(f"=== {yolov8_p2_path.name} (Mẫu tham chiếu chính thức của P2) ===")
    print(p2_yaml_content[:600] + "\n... [xem tiếp trong file] ...")

=== yolov8.yaml (Đầu vào của Baseline B0) ===
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLOv8 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolov8
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolov8n.yaml' will call yolov8.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024] # YOLOv8n summary: 129 layers, 3157200 parameters, 3157184 gradients, 8.9 GFLOPS
  s: [0.33, 0.50, 1024] # YOLOv8s summary: 129 layers, 11166560 
... [xem tiếp trong file] ...

=== yolov8-p2.yaml (Mẫu tham chiếu chính thức của P2) ===
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLOv8 object detection model with P2/4 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolov8
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters


### 5. Tạo cấu hình chuẩn `configs/yolov8n-p2.yaml`
Các yêu cầu kỹ thuật đối với `configs/yolov8n-p2.yaml`:
1. `nc: 6`: Khai báo 6 lớp khuyết tật PCB (missing_hole, mouse_bite, open_circuit, short, spur, spurious_copper).
2. `scales`: Chọn scale `n` (nano) đồng bộ tuyệt đối với mô hình gốc $B_0$ (`depth: 0.33, width: 0.25, max_channels: 1024`).
3. `head`: Có tầng Detect trên 4 mức phân giải `[18, 21, 24, 27]` tương ứng `[P2, P3, P4, P5]`.
4. **Không thêm cơ chế chú ý (attention)** và **không đổi công thức loss**.

In [4]:
p2_yaml_content = """# Ultralytics YOLOv8n-P2 configuration for PCB Defect Detection (KLTN PCB - Week 7-8)
# Architecture: High-resolution P2 layer added for tiny defect detection
# Feature map outputs: P2 (stride 4, 160x160) - P5 (stride 32, 20x20) at input 640x640

# Parameters
nc: 6 # 6 PCB defect classes: missing_hole, mouse_bite, open_circuit, short, spur, spurious_copper
scales: # model compound scaling constants for 'n' (YOLOv8n)
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]

# YOLOv8 backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]    # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]   # 1-P2/4
  - [-1, 3, C2f, [128, True]]    # 2
  - [-1, 1, Conv, [256, 3, 2]]   # 3-P3/8
  - [-1, 6, C2f, [256, True]]    # 4
  - [-1, 1, Conv, [512, 3, 2]]   # 5-P4/16
  - [-1, 6, C2f, [512, True]]    # 6
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]   # 8
  - [-1, 1, SPPF, [1024, 5]]     # 9

# YOLOv8-P2 head with 4 detection scales [P2, P3, P4, P5]
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 10
  - [[-1, 6], 1, Concat, [1]]                   # 11 cat backbone P4
  - [-1, 3, C2f, [512]]                         # 12

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 13
  - [[-1, 4], 1, Concat, [1]]                   # 14 cat backbone P3
  - [-1, 3, C2f, [256]]                         # 15 (P3/8-small)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 16
  - [[-1, 2], 1, Concat, [1]]                   # 17 cat backbone P2
  - [-1, 3, C2f, [128]]                         # 18 (P2/4-tiny)

  - [-1, 1, Conv, [128, 3, 2]]                  # 19
  - [[-1, 15], 1, Concat, [1]]                  # 20 cat head P3
  - [-1, 3, C2f, [256]]                         # 21 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]                  # 22
  - [[-1, 12], 1, Concat, [1]]                  # 23 cat head P4
  - [-1, 3, C2f, [512]]                         # 24 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]                  # 25
  - [[-1, 9], 1, Concat, [1]]                   # 26 cat head P5
  - [-1, 3, C2f, [1024]]                        # 27 (P5/32-large)

  - [[18, 21, 24, 27], 1, Detect, [nc]]         # 28 Detect(P2, P3, P4, P5)
"""

target_config_path = ROOT / "configs" / "yolov8n-p2.yaml"
target_config_path.parent.mkdir(parents=True, exist_ok=True)
with open(target_config_path, "w", encoding="utf-8") as f:
    f.write(p2_yaml_content.strip() + "\n")

print(f"Đã lưu thành công cấu hình kiến trúc B1 vào: {target_config_path}")

Đã lưu thành công cấu hình kiến trúc B1 vào: /Users/Project/kltn-pcb/configs/yolov8n-p2.yaml


### 6. Kiểm tra file output và thẩm định cấu trúc Detect Head
- Xác nhận `configs/yolov8n-p2.yaml` tồn tại.
- Đọc và in cấu trúc phần `head`.
- Khởi tạo thử nghiệm mô hình bằng `YOLO("configs/yolov8n-p2.yaml")` để đảm bảo parser của Ultralytics tương thích 100% không phát sinh lỗi cú pháp.

In [5]:
assert target_config_path.exists(), f"LỖI: Không tìm thấy {target_config_path}"

with open(target_config_path, "r", encoding="utf-8") as f:
    config_data = yaml.safe_load(f)

print("[XÁC MINH CẤU TRÚC]")
print(f"  - Số lớp nc       : {config_data.get('nc')}")
print(f"  - Scale định nghĩa: {list(config_data.get('scales', {}).keys())}")
print(f"  - Số layer backbone: {len(config_data.get('backbone', []))}")
print(f"  - Số layer head    : {len(config_data.get('head', []))}")

last_head_layer = config_data.get('head', [])[-1]
print(f"  - Lớp Detect cuối : {last_head_layer}")

detect_inputs = last_head_layer[0]
assert len(detect_inputs) == 4, f"LỖI: Detect head phải có đúng 4 đầu vào [P2, P3, P4, P5], thực tế nhận {detect_inputs}"
print(f"  -> [PASS] Detect head nhận đúng 4 đầu vào feature map: {detect_inputs}\n")

# Thử nghiệm khởi tạo mô hình bằng Ultralytics YOLO
print("Đang khởi tạo thử nghiệm cấu trúc mô hình qua Ultralytics...")
test_model = YOLO(str(target_config_path))
detect_module = test_model.model.model[-1]
print(f"Mô hình khởi tạo thành công: {type(test_model.model).__name__}")
print(f"Loại Detect module: {type(detect_module).__name__}")
print(f"Số lượng đầu ra dự đoán (nl): {detect_module.nl} mức phân giải")

[XÁC MINH CẤU TRÚC]
  - Số lớp nc       : 6
  - Scale định nghĩa: ['n']
  - Số layer backbone: 10
  - Số layer head    : 19
  - Lớp Detect cuối : [[18, 21, 24, 27], 1, 'Detect', ['nc']]
  -> [PASS] Detect head nhận đúng 4 đầu vào feature map: [18, 21, 24, 27]

Đang khởi tạo thử nghiệm cấu trúc mô hình qua Ultralytics...
Mô hình khởi tạo thành công: DetectionModel
Loại Detect module: Detect
Số lượng đầu ra dự đoán (nl): 4 mức phân giải


### 7. Ghi chú cơ sở khoa học dùng cho báo cáo luận văn

| Tầng đặc trưng | Stride | Độ phân giải feature map (với input $640 \times 640$) | Kích thước khuyết tật $20 \times 20$ px trên feature map | Vai trò trong phát hiện lỗi PCB |
| :--- | :---: | :---: | :---: | :--- |
| **$P_2$** *(Mới ở B1)* | **4** | **$160 \times 160$** | **$5.0 \times 5.0$ ô (cells)** | **Chuyên trách lỗi cực nhỏ (tiny/small): `mouse_bite`, `spur`, vết xước vi mô** |
| **$P_3$** | 8 | $80 \times 80$ | $2.5 \times 2.5$ ô (cells) | Lỗi nhỏ thông thường | 
| **$P_4$** | 16 | $40 \times 40$ | $1.25 \times 1.25$ ô (cells) | Lỗi kích thước trung bình (medium) |
| **$P_5$** | 32 | $20 \times 20$ | $0.625 \times 0.625$ ô (cells) | Ngữ cảnh toàn cục (global context) |

#### Phân tích định lượng:
- **Vấn đề của B0 tại $P_3$**: Với độ phân giải đầu vào $640 \times 640$, một khuyết tật kích thước $20 \times 20$ pixel chỉ chiếm **$2.5 \times 2.5$ ô** trên lưới đặc trưng của tầng $P_3$. Ở mức này, sau nhiều lớp tích chập gom đặc trưng, tín hiệu biên dạng rất dễ bị lẫn vào nền đường mạch đồng, dẫn đến hiện tượng bỏ sót (False Negative cao).
- **Ưu thế của B1 khi có tầng $P_2$**: Tầng $P_2$ bảo toàn kích thước không gian ở mức **$160 \times 160$** (stride 4). Khuyết tật $20 \times 20$ pixel trên $P_2$ chiếm tới **$5.0 \times 5.0$ ô**, diện tích thụ cảm và độ nét biên tăng gấp 4 lần so với $P_3$, cho phép mạng bảo toàn chi tiết biên cạnh vi mô mà không làm mất thông tin ngữ cảnh.